Used to prove that simple arbs do exist, with out fees, or taxes

In [6]:
import pandas as pd
import requests
import json

Get Order book data

In [7]:
def get_raw_dfs(match_df):
    records = match_df.to_dict('records')
    poly_raw_list = []
    kalshi_raw_list = []
    
    # Use a session for faster consecutive requests
    session = requests.Session()

    for row in records:
        m_id = row.get('match_id')
        ids = row.get('market_ids', {})
        
        # --- 1. POLYMARKET ---
        poly_id = ids.get('poly')
        if poly_id:
            try:
                m_res = session.get(f"https://gamma-api.polymarket.com/markets/{poly_id}", timeout=5)
                if m_res.status_code == 200:
                    m_data = m_res.json()
                    tokens = json.loads(m_data.get('clobTokenIds', '[]'))
                    if tokens:
                        book_res = session.get(f"https://clob.polymarket.com/book?token_id={tokens[0]}", timeout=5)
                        if book_res.status_code == 200:
                            full_poly = book_res.json()
                            full_poly['question'] = m_data.get('question')
                            full_poly['match_id'] = m_id
                            poly_raw_list.append(full_poly)
            except Exception as e: print(f"Poly Error ({poly_id}): {e}")

        # --- 2. KALSHI ---
        kalshi_id = ids.get('kalshi')
        if kalshi_id:
            try:
                # Get Orderbook
                k_book_url = f"https://api.elections.kalshi.com/trade-api/v2/markets/{kalshi_id}/orderbook"
                k_res = session.get(k_book_url, timeout=5)
                
                # NEW: Get Market Metadata (for the Question/Title)
                k_meta_url = f"https://api.elections.kalshi.com/trade-api/v2/markets/{kalshi_id}"
                k_meta_res = session.get(k_meta_url, timeout=5)
                
                if k_res.status_code == 200:
                    k_data = k_res.json()
                    k_data['match_id'] = m_id
                    k_data['market_id'] = kalshi_id
                    
                    if k_meta_res.status_code == 200:
                        k_data['question'] = k_meta_res.json().get('market', {}).get('title')
                    
                    kalshi_raw_list.append(k_data)
            except Exception as e: print(f"Kalshi Error ({kalshi_id}): {e}")

    df_poly = pd.json_normalize(poly_raw_list) if poly_raw_list else pd.DataFrame()
    df_kalshi = pd.json_normalize(kalshi_raw_list) if kalshi_raw_list else pd.DataFrame()
    return df_kalshi, df_poly


arb_candidates = pd.read_json('data/arb_candidates.json')
print("Fetching raw data for arbitrage candidates...")
kalshi_df_raw, poly_df_raw = get_raw_dfs(arb_candidates)

Cleaning

In [10]:
def clean_poly_book(row, side='bid'):
    data = row.get('bids' if side == 'bid' else 'asks')
    if not isinstance(data, list) or not data: return None
    
    # Sort and convert to float
    rev = (side == 'bid')
    sorted_list = sorted(data, key=lambda x: float(x['price']), reverse=rev)
    return float(sorted_list[0]['price'])

if not poly_df_raw.empty:
    poly_df_raw = poly_df_raw.rename(columns={'market': 'market_id'})
    poly_df = poly_df_raw[['match_id', 'market_id', 'question', 'bids', 'asks']].copy()
    poly_df['best_bid'] = poly_df.apply(lambda x: clean_poly_book(x, 'bid'), axis=1)
    poly_df['best_ask'] = poly_df.apply(lambda x: clean_poly_book(x, 'ask'), axis=1)
    poly_df = poly_df.drop(columns=['bids', 'asks'])
else:
    poly_df = pd.DataFrame()


def get_kalshi_prices(row):
    no_book = row.get('orderbook_fp.no_dollars', [])
    yes_book = row.get('orderbook_fp.yes_dollars', [])
    res = {'best_bid_yes': None, 'best_ask_yes': None}
    
    if yes_book:
        res['best_bid_yes'] = float(sorted(yes_book, key=lambda x: float(x[0]), reverse=True)[0][0])
    if no_book:
        best_no_bid = float(sorted(no_book, key=lambda x: float(x[0]), reverse=True)[0][0])
        res['best_ask_yes'] = round(1.0 - best_no_bid, 4)
    return pd.Series(res)

if not kalshi_df_raw.empty:
    # Capture the new kalshi_question here
    k_cols = ['match_id', 'market_id', 'question']
    kalshi_df = kalshi_df_raw[[c for c in k_cols if c in kalshi_df_raw.columns]].copy()
    kalshi_df[['best_bid', 'best_ask']] = kalshi_df_raw.apply(get_kalshi_prices, axis=1)
else:
    kalshi_df = pd.DataFrame()


Arb Calculation

In [9]:
if not poly_df.empty and not kalshi_df.empty:

    combined_df = pd.merge(poly_df, kalshi_df, on='match_id', suffixes=('_poly', '_kalshi'))

    # Calculations
    combined_df['profit_poly_to_kal'] = combined_df['best_bid_kalshi'] - combined_df['best_ask_poly']
    combined_df['profit_kal_to_poly'] = combined_df['best_bid_poly'] - combined_df['best_ask_kalshi']
    
    # Filter for profit > 0 (or add a 0.01 buffer for fees)
    buffer = 0.005 # 0.5 cent buffer
    arbs = combined_df[
        (combined_df['profit_poly_to_kal'] > buffer) | 
        (combined_df['profit_kal_to_poly'] > buffer)
    ].copy()
    
    # Add a summary column to see which way to trade
    arbs['best_move'] = arbs.apply(lambda r: 'Buy Poly/ Sell Kalshi' if r['profit_poly_to_kal'] > r['profit_kal_to_poly'] else 'Buy Kalshi/ Sell Poly', axis=1)
    
    print(f"Found {len(arbs)} arb opportunities!")
    display(arbs)
else:
    print("One or both dataframes are empty. Check API connections.")

Found 8 arb opportunities!


,match_id,market_id_poly,question_poly,best_bid_poly,best_ask_poly,market_id_kalshi,question_kalshi,best_bid_kalshi,best_ask_kalshi,profit_poly_to_kal,profit_kal_to_poly,best_move
0,037fcde160,0xdcae9573d5680a2c958cca4676ed28df7a0686157212...,Will Bitcoin outperform Gold in 2026?,0.330,0.350,KXBTCVSGOLD-26,Will Bitcoin outperform gold in 2026?,0.363,0.368,0.013,-0.038,Buy Poly/ Sell Kalshi
1,46f82f47c0,0x4618b3d681487cd0eb31bb82eb2862bcb572eea220af...,Will Olivia Rodrigo release an album in 2026?,0.960,0.985,KXALBUMRELEASE-26-OLIV,Will Olivia Rodrigo release a new album in 2026?,0.990,NaN,0.005,NaN,Buy Kalshi/ Sell Poly
5,381f0c05ae,0x6a07dfa017ff96c5502d8c2e64cb00b0bdeb8a8426ec...,Will Rihanna release an album in 2026?,0.300,0.330,KXALBUMRELEASE-26-RIH,Will Rihanna release a new album in 2026?,0.210,0.280,-0.120,0.020,Buy Kalshi/ Sell Poly
9,ddd0582f43,0xbf97a4643151b6eaab647a112be686a52133a5f6af3e...,Will Kendrick Lamar release an album in 2026?,0.370,0.490,KXALBUMRELEASE-26-KEN,Will Kendrick Lamar release a new album in 2026?,0.290,0.350,-0.200,0.020,Buy Kalshi/ Sell Poly
10,def2f99598,0x57630ca87de434c74db1c388ed88c1920a6d1dfcf409...,Katy Perry and Justin Trudeau engaged by end o...,0.270,0.300,KXENGAGEMENTTRUDEAUPERRY-26,Will Katy Perry and Justin Trudeau be engaged ...,0.340,0.350,0.040,-0.080,Buy Poly/ Sell Kalshi
13,81f19a5d91,0xbc5010c6f08146d966cb776dcd79c302168b583e8bce...,Will Alnylam Pharmaceuticals (ALNY) be added t...,0.001,0.004,KXSP500ADDQ-26MAR31-ALNY,Will Alnylam Pharmaceuticals be added to S&P 5...,0.010,0.020,0.006,-0.019,Buy Poly/ Sell Kalshi
19,ae80c334ef,0x5c1e362faefe4c2b65eaabd57422cbe25b0508ccee8e...,Will Affirm Holdings (AFRM) be added to the S&...,0.001,0.002,KXSP500ADDQ-26MAR31-AFRM,Will Affirm Holdings be added to S&P 500 befor...,0.010,0.020,0.008,-0.019,Buy Poly/ Sell Kalshi
20,6f87bef346,0xa6ef52567677573f60cc0dd4450d9c5bc0bc67fac27b...,Will Greenland vote for independence in 2026?,0.070,0.080,KXGREENIND-27,Will Greenland vote for independence before 2027?,0.050,0.060,-0.030,0.010,Buy Kalshi/ Sell Poly
